In [1]:
!pip install -q transformers                                                                                                                                                                                                                   
!pip install -q peft                                                                                                                                                                                                                           
!pip install -q bitsandbytes                                                                                                                                                                                                                   
!pip install -q datasets                                                                                                                                                                                                                       
!pip install -q accelerate                                                                                                                                                                                                                     
!pip install -q trl  

In [2]:
import torch                                                                                                                                                                                                                                   
print("GPU Available:", torch.cuda.is_available())                                                                                                                                                                                             
print("GPU Name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

GPU Available: True
GPU Name: Tesla T4


In [5]:
from huggingface_hub import login                                                                                                                                                                                                              
login() 

In [6]:
import torch                                                                                                                                                                                                                                   
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig                                                                                                                                                               
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training                                                                                                                                                                   
from datasets import load_dataset                                                                                                                                                                                                              
from trl import SFTTrainer, SFTConfig 

In [7]:
# Model name                                                                                                                                                                                                                                   
model_name = "deepseek-ai/deepseek-coder-1.3b-base"                                                                                                                                                                                            
                                                                                                                                                                                                                                                 
# 4-bit quantization settings                                                                                                                                                                                                                  
bnb_config = BitsAndBytesConfig(                                                                                                                                                                                                               
      load_in_4bit=True,                                                                                                                                                                                                                         
      bnb_4bit_quant_type="nf4",                                                                                                                                                                                                                 
      bnb_4bit_compute_dtype=torch.float16                                                                                                                                                                                                       
)                                                                                                                                                                                                                                              
                                                                                                                                                                                                                                                 
# Load tokenizer (converts text to numbers)                                                                                                                                                                                                    
print("Loading tokenizer...")                                                                                                                                                                                                                  
tokenizer = AutoTokenizer.from_pretrained(model_name)                                                                                                                                                                                          
tokenizer.pad_token = tokenizer.eos_token                                                                                                                                                                                                      
                                                                                                                                                                                                                                                 
# Load model in 4-bit                                                                                                                                                                                                                          
print("Loading model (this takes 1-2 minutes)...")                                                                                                                                                                                             
model = AutoModelForCausalLM.from_pretrained(                                                                                                                                                                                                  
      model_name,                                                                                                                                                                                                                                
      quantization_config=bnb_config,                                                                                                                                                                                                            
      device_map="auto"                                                                                                                                                                                                                          
)                                                                                                                                                                                                                                              
                                                                                                                                                                                                                                                 
print("Model loaded!")     

Loading tokenizer...
Loading model (this takes 1-2 minutes)...


Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

Model loaded!


In [8]:
# Prepare model for training                                                                                                                                                                                                                   
model = prepare_model_for_kbit_training(model)                                                                                                                                                                                                 
                                                                                                                                                                                                                                                 
  # LoRA settings (remember these?)                                                                                                                                                                                                              
lora_config = LoraConfig(                                                                                                                                                                                                                      
      r=16,                     # Rank - size of adapter                                                                                                                                                                                         
      lora_alpha=32,            # Alpha - 2x rank                                                                                                                                                                                                
      lora_dropout=0.05,        # Dropout - prevents overfitting                                                                                                                                                                                 
      bias="none",                                                                                                                                                                                                                               
      task_type="CAUSAL_LM",                                                                                                                                                                                                                     
      target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]  # Which layers to adapt                                                                                                                                                           
  )                                                                                                                                                                                                                                              
                                                                                                                                                                                                                                                 
# Add LoRA to model                                                                                                                                                                                                                            
model = get_peft_model(model, lora_config)                                                                                                                                                                                                     
                                                                                                                                                                                                                                                 
# Show trainable parameters                                                                                                                                                                                                                    
model.print_trainable_parameters() 

trainable params: 6,291,456 || all params: 1,352,763,392 || trainable%: 0.4651


In [9]:
dataset = load_dataset("Clinton/Text-to-sql-v1", split="train")                                                                                                                                                                                
                                                                                                                                                                                                                                                 
print(f"Total examples: {len(dataset)}")                                                                                                                                                                                                       
print("\nSample example:")                                                                                                                                                                                                                     
print(dataset[0])

Total examples: 262208

Sample example:
{'instruction': 'Name the home team for carlton away team', 'input': 'CREATE TABLE table_name_77 (\n    home_team VARCHAR,\n    away_team VARCHAR\n)', 'response': 'SELECT home_team FROM table_name_77 WHERE away_team = "carlton"', 'source': 'sql_create_context', 'text': 'Below are sql tables schemas paired with instruction that describes a task. Using valid SQLite, write a response that appropriately completes the request for the provided tables. ### Instruction: Name the home team for carlton away team ### Input: CREATE TABLE table_name_77 (\n    home_team VARCHAR,\n    away_team VARCHAR\n) ### Response: SELECT home_team FROM table_name_77 WHERE away_team = "carlton"'}


In [10]:
import random                                                                                                                                                                                                                                  
                                                                                                                                                                                                                                                 
small_dataset = dataset.shuffle(seed=42).select(range(500))                                                                                                                                                                                    
                                                                                                                                                                                                                                                 
def format_example(example):                                                                                                                                                                                                                   
      text = f"""### Instruction:                                                                                                                                                                                                                
Convert this question to SQL.                                                                                                                                                                                                                  
                                                                                                                                                                                                                                                 
### Input:                                                                                                                                                                                                                                     
{example['instruction']}                                                                                                                                                                                                                       
{example['input']}                                                                                                                                                                                                                             
                                                                                                                                                                                                                                                 
### Response:                                                                                                                                                                                                                                  
{example['response']}"""                                                                                                                                                                                                                       
      return {"text": text}                                                                                                                                                                                                                      
                                                                                                                                                                                                                                                 
train_dataset = small_dataset.map(format_example)                                                                                                                                                                                              
                                                                                                                                                                                                                                                 
print(f"Training examples: {len(train_dataset)}")                                                                                                                                                                                              
print("\nFormatted example:")                                                                                                                                                                                                                  
print(train_dataset[0]['text'][:500])

Training examples: 500

Formatted example:
### Instruction:                                                                                                                                                                                                                
Convert this question to SQL.                                                                                                                                                                                                                  
                                   


In [11]:
from transformers import TrainingArguments, Trainer                                                                                                                                                                                            
                                                                                                                                                                                                                                                 
training_args = TrainingArguments(                                                                                                                                                                                                             
      output_dir="./deepseek-sql-lora",                                                                                                                                                                                                          
      num_train_epochs=1,                                                                                                                                                                                                                        
      per_device_train_batch_size=2,                                                                                                                                                                                                             
      gradient_accumulation_steps=4,                                                                                                                                                                                                             
      learning_rate=2e-4,                                                                                                                                                                                                                        
      logging_steps=10,                                                                                                                                                                                                                          
      save_strategy="epoch",                                                                                                                                                                                                                     
      fp16=True,                                                                                                                                                                                                                                 
)                                                                                                                                                                                                                                              
                                                                                                                                                                                                                                                 
def tokenize_function(example):                                                                                                                                                                                                                
      return tokenizer(                                                                                                                                                                                                                          
          example["text"],                                                                                                                                                                                                                       
          truncation=True,                                                                                                                                                                                                                       
          max_length=512,                                                                                                                                                                                                                        
          padding="max_length"                                                                                                                                                                                                                   
      )                                                                                                                                                                                                                                          
                                                                                                                                                                                                                                                 
tokenized_dataset = train_dataset.map(tokenize_function, batched=True)                                                                                                                                                                         
tokenized_dataset = tokenized_dataset.remove_columns(["text", "instruction", "input", "response"])                                                                                                                                             
                                                                                                                                                                                                                                                 
import copy                                                                                                                                                                                                                                    
tokenized_dataset = tokenized_dataset.map(lambda x: {"labels": copy.copy(x["input_ids"])})                                                                                                                                                     
                                                                                                                                                                                                                                                 
trainer = Trainer(                                                                                                                                                                                                                             
      model=model,                                                                                                                                                                                                                               
      args=training_args,                                                                                                                                                                                                                        
      train_dataset=tokenized_dataset,                                                                                                                                                                                                           
)                                                                                                                                                                                                                                              
                                                                                                                                                                                                                                                 
print("Starting training...")                                                                                                                                                                                                                  
trainer.train()                                                                                                                                                                                                                                
print("Training complete!")     

Starting training...


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,12.387061
20,0.731659
30,0.610080
40,0.557494
50,0.421443
60,0.407376


Training complete!


In [19]:
def ask_sql(question, context):                                                                                                                                                                                                                
    prompt = f"""### Instruction:                                                                                                                                                                                                              
{question}                                                                                                                                                                                                                                     
                                                                                                                                                                                                                                                 
### Input:                                                                                                                                                                                                                                     
{context}                                                                                                                                                                                                                                      
                                                                                                                                                                                                                                                 
### Response:                                                                                                                                                                                                                                  
"""                                                                                                                                                                                                                                            
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)                                                                                                                                                                           
                                                                                                                                                                                                                                                 
    with torch.no_grad():                                                                                                                                                                                                                      
        outputs = model.generate(                                                                                                                                                                                                              
            **inputs,                                                                                                                                                                                                                          
            max_new_tokens=256,                                                                                                                                                                                                                
            pad_token_id=tokenizer.eos_token_id                                                                                                                                                                                                
        )                                                                                                                                                                                                                                      
                                                                                                                                                                                                                                                 
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)                                                                                                                                                                          
                                                                                                                                                                                                                                                 
    # Extract response after "### Response:"                                                                                                                                                                                                   
    if "### Response:" in response:                                                                                                                                                                                                            
        return response.split("### Response:")[-1].strip()                                                                                                                                                                                     
    return response.strip()                   

In [20]:
import math                                                                                                                                                                                                                                    
                                                                                                                                                                                                                                                 
def calculate_token_accuracy(expected, predicted):                                                                                                                                                                                             
    expected_tokens = expected.lower().split()                                                                                                                                                                                                 
    predicted_tokens = predicted.lower().split()                                                                                                                                                                                               
    matches = sum(1 for e, p in zip(expected_tokens, predicted_tokens) if e == p)                                                                                                                                                              
    max_len = max(len(expected_tokens), len(predicted_tokens))                                                                                                                                                                                 
    return (matches / max_len) * 100 if max_len > 0 else 0                                                                                                                                                                                     
                                                                                                                                                                                                                                                 
test_dataset = dataset.shuffle(seed=123).select(range(500, 550))                                                                                                                                                                               
                                                                                                                                                                                                                                                 
exact_matches = 0                                                                                                                                                                                                                              
total_token_accuracy = 0                                                                                                                                                                                                                       
errors = []                                                                                                                                                                                                                                    
token_accuracies = []                                                                                                                                                                                                                          
total = len(test_dataset)                                                                                                                                                                                                                      
                                                                                                                                                                                                                                                 
print("=" * 60)                                                                                                                                                                                                                                
print(f"EVALUATING ON {total} TEST EXAMPLES")                                                                                                                                                                                                  
print("=" * 60)                                                                                                                                                                                                                                
                                                                                                                                                                                                                                                 
for i, example in enumerate(test_dataset):                                                                                                                                                                                                     
    context = example['input'] if example['input'] else ""                                                                                                                                                                                     
    question = example['instruction']                                                                                                                                                                                                          
    expected = example['response']                                                                                                                                                                                                             
    predicted = ask_sql(question, context)                                                                                                                                                                                                     
    is_exact = predicted.strip().lower() == expected.strip().lower()                                                                                                                                                                           
    token_acc = calculate_token_accuracy(expected, predicted)                                                                                                                                                                                  
    error = 1 - (token_acc / 100)                                                                                                                                                                                                              
    if is_exact:                                                                                                                                                                                                                               
        exact_matches += 1                                                                                                                                                                                                                     
    total_token_accuracy += token_acc                                                                                                                                                                                                          
    token_accuracies.append(token_acc)                                                                                                                                                                                                         
    errors.append(error)                                                                                                                                                                                                                       
    if (i + 1) % 10 == 0:                                                                                                                                                                                                                      
        print(f"Processed {i+1}/{total}...")                                                                                                                                                                                                   
                                                                                                                                                                                                                                                 
exact_match_pct = (exact_matches / total) * 100                                                                                                                                                                                                
avg_token_acc = total_token_accuracy / total                                                                                                                                                                                                   
mean_error = sum(errors) / len(errors)                                                                                                                                                                                                         
mse = sum(e ** 2 for e in errors) / len(errors)                                                                                                                                                                                                
rmse = math.sqrt(mse)                                                                                                                                                                                                                          
                                                                                                                                                                                                                                                 
mean_acc = sum(token_accuracies) / len(token_accuracies)                                                                                                                                                                                       
ss_tot = sum((acc - mean_acc) ** 2 for acc in token_accuracies)                                                                                                                                                                                
ss_res = sum((acc - 100) ** 2 for acc in token_accuracies)                                                                                                                                                                                     
r_squared = 1 - (ss_res / ss_tot) if ss_tot != 0 else 0                                                                                                                                                                                        
                                                                                                                                                                                                                                                 
print("\n" + "=" * 60)                                                                                                                                                                                                                         
print("EVALUATION SUMMARY")                                                                                                                                                                                                                    
print("=" * 60)                                                                                                                                                                                                                                
print(f"Exact Match Accuracy:   {exact_match_pct:.2f}%")                                                                                                                                                                                       
print(f"Mean Token Accuracy:    {avg_token_acc:.2f}%")                                                                                                                                                                                         
print(f"Mean Error:             {mean_error:.4f}")                                                                                                                                                                                             
print(f"MSE:                    {mse:.4f}")                                                                                                                                                                                                    
print(f"RMSE:                   {rmse:.4f}")                                                                                                                                                                                                   
print(f"R-Squared:              {r_squared:.4f}")                                                                                                                                                                                              
print(f"Total Examples:         {total}")                                                                                                                                                                                                      
print(f"Exact Matches:          {exact_matches}")                                                                                                                                                                                              
                                             

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
Caching is incompatible with gradient checkpointing in LlamaDecoderLayer. Setting `past_key_values=None`.


EVALUATING ON 50 TEST EXAMPLES


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


Processed 10/50...
Processed 20/50...
Processed 30/50...
Processed 40/50...
Processed 50/50...

EVALUATION SUMMARY
Exact Match Accuracy:   0.00%
Mean Token Accuracy:    0.30%
Mean Error:             0.9970
MSE:                    0.9940
RMSE:                   0.9970
R-Squared:              -38208.6674
Total Examples:         50
Exact Matches:          0


In [21]:
print("Reloading fresh model...")                                                                                                                                                                                                              
                                                                                                                                                                                                                                                 
model = AutoModelForCausalLM.from_pretrained(                                                                                                                                                                                                  
    "deepseek-ai/deepseek-coder-1.3b-base",                                                                                                                                                                                                    
    quantization_config=bnb_config,                                                                                                                                                                                                            
    device_map="auto"                                                                                                                                                                                                                          
)                                                                                                                                                                                                                                              
                                                                                                                                                                                                                                                 
model = prepare_model_for_kbit_training(model)                                                                                                                                                                                                 
                                                                                                                                                                                                                                                 
lora_config = LoraConfig(                                                                                                                                                                                                                      
    r=32,                                                                                                                                                                                                                                      
    lora_alpha=64,                                                                                                                                                                                                                             
    lora_dropout=0.05,                                                                                                                                                                                                                         
    bias="none",                                                                                                                                                                                                                               
    task_type="CAUSAL_LM",                                                                                                                                                                                                                     
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]                                                                                                                                                                                    
)                                                                                                                                                                                                                                              
                                                                                                                                                                                                                                                 
model = get_peft_model(model, lora_config)                                                                                                                                                                                                     
model.print_trainable_parameters()                                                                                                                                                                                                             
                                                                                                                                                                                                                                                 
print("Fresh model ready!") 

Reloading fresh model...


Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

trainable params: 12,582,912 || all params: 1,359,054,848 || trainable%: 0.9259
Fresh model ready!


In [22]:
print("Preparing 2000 training examples...")                                                                                                                                                                                                   
                                                                                                                                                                                                                                                 
large_dataset = dataset.shuffle(seed=42).select(range(2000))                                                                                                                                                                                   
                                                                                                                                                                                                                                                 
def format_example(example):                                                                                                                                                                                                                   
    text = f"""### Instruction:                                                                                                                                                                                                                
Convert this question to SQL.                                                                                                                                                                                                                  
                                                                                                                                                                                                                                                 
### Input:                                                                                                                                                                                                                                     
{example['instruction']}                                                                                                                                                                                                                       
{example['input']}                                                                                                                                                                                                                             
                                                                                                                                                                                                                                                 
### Response:                                                                                                                                                                                                                                  
{example['response']}"""                                                                                                                                                                                                                       
    return {"text": text}                                                                                                                                                                                                                      
                                                                                                                                                                                                                                                 
train_dataset = large_dataset.map(format_example)                                                                                                                                                                                              
                                                                                                                                                                                                                                                 
def tokenize_function(example):                                                                                                                                                                                                                
    return tokenizer(                                                                                                                                                                                                                          
        example["text"],                                                                                                                                                                                                                       
        truncation=True,                                                                                                                                                                                                                       
        max_length=512,                                                                                                                                                                                                                        
        padding="max_length"                                                                                                                                                                                                                   
    )                                                                                                                                                                                                                                          
                                                                                                                                                                                                                                                 
tokenized_dataset = train_dataset.map(tokenize_function, batched=True)                                                                                                                                                                         
tokenized_dataset = tokenized_dataset.remove_columns(["text", "instruction", "input", "response"])                                                                                                                                             
                                                                                                                                                                                                                                                 
import copy                                                                                                                                                                                                                                    
tokenized_dataset = tokenized_dataset.map(lambda x: {"labels": copy.copy(x["input_ids"])})                                                                                                                                                     
                                                                                                                                                                                                                                                 
print(f"Training examples ready: {len(tokenized_dataset)}") 

Preparing 2000 training examples...


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Training examples ready: 2000


In [23]:
from transformers import TrainingArguments, Trainer                                                                                                                                                                                            
                                                                                                                                                                                                                                                 
training_args = TrainingArguments(                                                                                                                                                                                                             
    output_dir="./deepseek-sql-lora-v2",                                                                                                                                                                                                       
    num_train_epochs=3,                                                                                                                                                                                                                        
    per_device_train_batch_size=4,                                                                                                                                                                                                             
    gradient_accumulation_steps=4,                                                                                                                                                                                                             
    learning_rate=2e-4,                                                                                                                                                                                                                        
    logging_steps=50,                                                                                                                                                                                                                          
    save_strategy="epoch",                                                                                                                                                                                                                     
    fp16=True,                                                                                                                                                                                                                                 
    warmup_steps=50,                                                                                                                                                                                                                           
)                                                                                                                                                                                                                                              
                                                                                                                                                                                                                                                 
trainer = Trainer(                                                                                                                                                                                                                             
    model=model,                                                                                                                                                                                                                               
    args=training_args,                                                                                                                                                                                                                        
    train_dataset=tokenized_dataset,                                                                                                                                                                                                           
)                                                                                                                                                                                                                                              
                                                                                                                                                                                                                                                 
print("Starting training (3 epochs, 2000 examples)...")                                                                                                                                                                                        
print("This will take ~15-20 minutes...")                                                                                                                                                                                                      
trainer.train()                                                                                                                                                                                                                                
print("Training complete!") 

Starting training (3 epochs, 2000 examples)...
This will take ~15-20 minutes...


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
50,6.751349
100,0.366252
150,0.312524
200,0.292965
250,0.250903
300,0.241001
350,0.237662


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Training complete!


In [24]:
model.eval()                                                                                                                                                                                                                                   
model.config.use_cache = True                                                                                                                                                                                                                  
                                                                                                                                                                                                                                                 
def ask_sql(question, context=""):                                                                                                                                                                                                             
    prompt = f"""### Instruction:                                                                                                                                                                                                              
Convert this question to SQL.                                                                                                                                                                                                                  
                                                                                                                                                                                                                                                 
### Input:                                                                                                                                                                                                                                     
{question}                                                                                                                                                                                                                                     
{context}                                                                                                                                                                                                                                      
                                                                                                                                                                                                                                                 
### Response:                                                                                                                                                                                                                                  
"""                                                                                                                                                                                                                                            
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)                                                                                                                                                                           
    with torch.no_grad():                                                                                                                                                                                                                      
        outputs = model.generate(                                                                                                                                                                                                              
            **inputs,                                                                                                                                                                                                                          
            max_new_tokens=100,                                                                                                                                                                                                                
            do_sample=False,                                                                                                                                                                                                                   
            repetition_penalty=1.2,                                                                                                                                                                                                            
            pad_token_id=tokenizer.eos_token_id                                                                                                                                                                                                
        )                                                                                                                                                                                                                                      
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)                                                                                                                                                                          
    sql = response.split("### Response:")[-1].strip()                                                                                                                                                                                          
    sql = sql.split("\n")[0]                                                                                                                                                                                                                   
    return sql                                                                                                                                                                                                                                 
                                                                                                                                                                                                                                                 
result = ask_sql("Show all customers from New York", "CREATE TABLE customers (id INT, name VARCHAR, city VARCHAR)")                                                                                                                            
print("Test: Show all customers from New York")                                                                                                                                                                                                
print(f"SQL: {result}")   

Test: Show all customers from New York
SQL: SELECT * FROM customers WHERE CITY = "new york"


In [25]:
import math                                                                                                                                                                                                                                    
                                                                                                                                                                                                                                                 
def calculate_token_accuracy(expected, predicted):                                                                                                                                                                                             
    expected_tokens = expected.lower().split()                                                                                                                                                                                                 
    predicted_tokens = predicted.lower().split()                                                                                                                                                                                               
    matches = sum(1 for e, p in zip(expected_tokens, predicted_tokens) if e == p)                                                                                                                                                              
    max_len = max(len(expected_tokens), len(predicted_tokens))                                                                                                                                                                                 
    return (matches / max_len) * 100 if max_len > 0 else 0                                                                                                                                                                                     
                                                                                                                                                                                                                                                 
test_dataset = dataset.shuffle(seed=123).select(range(2000, 2050))                                                                                                                                                                             
                                                                                                                                                                                                                                                 
exact_matches = 0                                                                                                                                                                                                                              
total_token_accuracy = 0                                                                                                                                                                                                                       
errors = []                                                                                                                                                                                                                                    
token_accuracies = []                                                                                                                                                                                                                          
total = len(test_dataset)                                                                                                                                                                                                                      
                                                                                                                                                                                                                                                 
print("=" * 60)                                                                                                                                                                                                                                
print(f"EVALUATING V2 ON {total} TEST EXAMPLES")                                                                                                                                                                                               
print("=" * 60)                                                                                                                                                                                                                                
                                                                                                                                                                                                                                                 
for i, example in enumerate(test_dataset):                                                                                                                                                                                                     
    context = example['input'] if example['input'] else ""                                                                                                                                                                                     
    question = example['instruction']                                                                                                                                                                                                          
    expected = example['response']                                                                                                                                                                                                             
    predicted = ask_sql(question, context)                                                                                                                                                                                                     
    is_exact = predicted.strip().lower() == expected.strip().lower()                                                                                                                                                                           
    token_acc = calculate_token_accuracy(expected, predicted)                                                                                                                                                                                  
    error = 1 - (token_acc / 100)                                                                                                                                                                                                              
    if is_exact:                                                                                                                                                                                                                               
        exact_matches += 1                                                                                                                                                                                                                     
    total_token_accuracy += token_acc                                                                                                                                                                                                          
    token_accuracies.append(token_acc)                                                                                                                                                                                                         
    errors.append(error)                                                                                                                                                                                                                       
    if (i + 1) % 10 == 0:                                                                                                                                                                                                                      
        print(f"Processed {i+1}/{total}...")                                                                                                                                                                                                   
                                                                                                                                                                                                                                                 
exact_match_pct = (exact_matches / total) * 100                                                                                                                                                                                                
avg_token_acc = total_token_accuracy / total                                                                                                                                                                                                   
mean_error = sum(errors) / len(errors)                                                                                                                                                                                                         
mse = sum(e ** 2 for e in errors) / len(errors)                                                                                                                                                                                                
rmse = math.sqrt(mse)                                                                                                                                                                                                                          
                                                                                                                                                                                                                                                 
mean_acc = sum(token_accuracies) / len(token_accuracies)                                                                                                                                                                                       
ss_tot = sum((acc - mean_acc) ** 2 for acc in token_accuracies)                                                                                                                                                                                
ss_res = sum((acc - 100) ** 2 for acc in token_accuracies)                                                                                                                                                                                     
r_squared = 1 - (ss_res / ss_tot) if ss_tot != 0 else 0                                                                                                                                                                                        
                                                                                                                                                                                                                                                 
print("\n" + "=" * 60)                                                                                                                                                                                                                         
print("EVALUATION SUMMARY - V2")                                                                                                                                                                                                               
print("=" * 60)                                                                                                                                                                                                                                
print(f"Exact Match Accuracy:   {exact_match_pct:.2f}%")                                                                                                                                                                                       
print(f"Mean Token Accuracy:    {avg_token_acc:.2f}%")                                                                                                                                                                                         
print(f"Mean Error:             {mean_error:.4f}")                                                                                                                                                                                             
print(f"MSE:                    {mse:.4f}")                                                                                                                                                                                                    
print(f"RMSE:                   {rmse:.4f}")                                                                                                                                                                                                   
print(f"R-Squared:              {r_squared:.4f}")                                                                                                                                                                                              
print(f"Total Examples:         {total}")                                                                                                                                                                                                      
print(f"Exact Matches:          {exact_matches}") 

EVALUATING V2 ON 50 TEST EXAMPLES
Processed 10/50...
Processed 20/50...
Processed 30/50...
Processed 40/50...
Processed 50/50...

EVALUATION SUMMARY - V2
Exact Match Accuracy:   2.00%
Mean Token Accuracy:    32.85%
Mean Error:             0.6715
MSE:                    0.5480
RMSE:                   0.7402
R-Squared:              -4.6482
Total Examples:         50
Exact Matches:          1


In [26]:
print("Loading fresh model (V2 settings, more data)...")                                                                                                                                                                                       
                                                                                                                                                                                                                                                 
model = AutoModelForCausalLM.from_pretrained(                                                                                                                                                                                                  
    "deepseek-ai/deepseek-coder-1.3b-base",                                                                                                                                                                                                    
    quantization_config=bnb_config,                                                                                                                                                                                                            
    device_map="auto"                                                                                                                                                                                                                          
)                                                                                                                                                                                                                                              
                                                                                                                                                                                                                                                 
model = prepare_model_for_kbit_training(model)                                                                                                                                                                                                 
                                                                                                                                                                                                                                                 
lora_config = LoraConfig(                                                                                                                                                                                                                      
    r=32,                                                                                                                                                                                                                                      
    lora_alpha=64,                                                                                                                                                                                                                             
    lora_dropout=0.05,                                                                                                                                                                                                                         
    bias="none",                                                                                                                                                                                                                               
    task_type="CAUSAL_LM",                                                                                                                                                                                                                     
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]                                                                                                                                                                                    
)                                                                                                                                                                                                                                              
                                                                                                                                                                                                                                                 
model = get_peft_model(model, lora_config)                                                                                                                                                                                                     
model.print_trainable_parameters()                                                                                                                                                                                                             
print("Model ready!") 

Loading fresh model (V2 settings, more data)...


Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

trainable params: 12,582,912 || all params: 1,359,054,848 || trainable%: 0.9259
Model ready!


In [27]:
print("Preparing 5000 training examples...")                                                                                                                                                                                                   
                                                                                                                                                                                                                                                 
large_dataset = dataset.shuffle(seed=42).select(range(5000))                                                                                                                                                                                   
                                                                                                                                                                                                                                                 
def format_example(example):                                                                                                                                                                                                                   
    text = f"""### Instruction:                                                                                                                                                                                                                
Convert this question to SQL.                                                                                                                                                                                                                  
                                                                                                                                                                                                                                                 
### Input:                                                                                                                                                                                                                                     
{example['instruction']}                                                                                                                                                                                                                       
{example['input']}                                                                                                                                                                                                                             
                                                                                                                                                                                                                                                 
### Response:                                                                                                                                                                                                                                  
{example['response']}"""                                                                                                                                                                                                                       
    return {"text": text}                                                                                                                                                                                                                      
                                                                                                                                                                                                                                                 
train_dataset = large_dataset.map(format_example)                                                                                                                                                                                              
                                                                                                                                                                                                                                                 
def tokenize_function(example):                                                                                                                                                                                                                
    return tokenizer(                                                                                                                                                                                                                          
        example["text"],                                                                                                                                                                                                                       
        truncation=True,                                                                                                                                                                                                                       
        max_length=512,                                                                                                                                                                                                                        
        padding="max_length"                                                                                                                                                                                                                   
    )                                                                                                                                                                                                                                          
                                                                                                                                                                                                                                                 
tokenized_dataset = train_dataset.map(tokenize_function, batched=True)                                                                                                                                                                         
tokenized_dataset = tokenized_dataset.remove_columns(["text", "instruction", "input", "response"])                                                                                                                                             
                                                                                                                                                                                                                                                 
import copy                                                                                                                                                                                                                                    
tokenized_dataset = tokenized_dataset.map(lambda x: {"labels": copy.copy(x["input_ids"])})                                                                                                                                                     
                                                                                                                                                                                                                                                 
print(f"Training examples ready: {len(tokenized_dataset)}")  

Preparing 5000 training examples...
Training examples ready: 5000


In [28]:
training_args = TrainingArguments(                                                                                                                                                                                                             
    output_dir="./deepseek-sql-lora-v2-plus",                                                                                                                                                                                                  
    num_train_epochs=5,                                                                                                                                                                                                                        
    per_device_train_batch_size=4,                                                                                                                                                                                                             
    gradient_accumulation_steps=4,                                                                                                                                                                                                             
    learning_rate=1e-4,                                                                                                                                                                                                                        
    logging_steps=100,                                                                                                                                                                                                                         
    save_strategy="epoch",                                                                                                                                                                                                                     
    fp16=True,                                                                                                                                                                                                                                 
    warmup_steps=100,                                                                                                                                                                                                                          
)                                                                                                                                                                                                                                              
                                                                                                                                                                                                                                                 
trainer = Trainer(                                                                                                                                                                                                                             
    model=model,                                                                                                                                                                                                                               
    args=training_args,                                                                                                                                                                                                                        
    train_dataset=tokenized_dataset,                                                                                                                                                                                                           
)                                                                                                                                                                                                                                              
                                                                                                                                                                                                                                                 
print("Starting training (5 epochs, 5000 examples)...")                                                                                                                                                                                        
print("This will take ~40-50 minutes...")                                                                                                                                                                                                      
trainer.train()                                                                                                                                                                                                                                
print("Training complete!") 

Starting training (5 epochs, 5000 examples)...
This will take ~40-50 minutes...


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
100,5.745079
200,0.321072
300,0.268152
400,0.239640
500,0.226944
600,0.211480
700,0.205641


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

Training complete!


In [30]:
 # Save locally                                                                                                                                                                                                                                 
model.save_pretrained("/kaggle/working/deepseek-sql-lora-v2")                                                                                                                                                                                  
tokenizer.save_pretrained("/kaggle/working/deepseek-sql-lora-v2")                                                                                                                                                                              
print("Model saved!")  

Model saved!


In [37]:
from peft import PeftModel                                                                                                                                                                                                                     
                                                                                                                                                                                                                                                 
base_model = AutoModelForCausalLM.from_pretrained(                                                                                                                                                                                             
    "deepseek-ai/deepseek-coder-1.3b-base",                                                                                                                                                                                                    
    quantization_config=bnb_config,                                                                                                                                                                                                            
    device_map="auto"                                                                                                                                                                                                                          
)                                                                                                                                                                                                                                              
                                                                                                                                                                                                                                                 
model = PeftModel.from_pretrained(                                                                                                                                                                                                             
    base_model,                                                                                                                                                                                                                                
    "/kaggle/working/deepseek-sql-lora-v2-plus/checkpoint-785"                                                                                                                                                                                 
)                                                                                                                                                                                                                                              
model.eval()                                                                                                                                                                                                                                   
print("Trained model loaded from checkpoint-785!")  

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

Trained model loaded from checkpoint-785!


In [38]:
import math                                                                                                                                                                                                                                    
                                                                                                                                                                                                                                                 
def calculate_token_accuracy(expected, predicted):                                                                                                                                                                                             
    expected_tokens = expected.lower().split()                                                                                                                                                                                                 
    predicted_tokens = predicted.lower().split()                                                                                                                                                                                               
    matches = sum(1 for e, p in zip(expected_tokens, predicted_tokens) if e == p)                                                                                                                                                              
    max_len = max(len(expected_tokens), len(predicted_tokens))                                                                                                                                                                                 
    return (matches / max_len) * 100 if max_len > 0 else 0                                                                                                                                                                                     
                                                                                                                                                                                                                                                 
test_dataset = dataset.shuffle(seed=123).select(range(5000, 5050))                                                                                                                                                                             
                                                                                                                                                                                                                                                 
exact_matches = 0                                                                                                                                                                                                                              
total_token_accuracy = 0                                                                                                                                                                                                                       
errors = []                                                                                                                                                                                                                                    
token_accuracies = []                                                                                                                                                                                                                          
total = len(test_dataset)                                                                                                                                                                                                                      
                                                                                                                                                                                                                                                 
print("=" * 60)                                                                                                                                                                                                                                
print(f"EVALUATING V2 ON {total} TEST EXAMPLES")                                                                                                                                                                                               
print("=" * 60)                                                                                                                                                                                                                                
                                                                                                                                                                                                                                                 
for i, example in enumerate(test_dataset):                                                                                                                                                                                                     
    context = example['input'] if example['input'] else ""                                                                                                                                                                                     
    question = example['instruction']                                                                                                                                                                                                          
    expected = example['response']                                                                                                                                                                                                             
    predicted = ask_sql(question, context)                                                                                                                                                                                                     
    is_exact = predicted.strip().lower() == expected.strip().lower()                                                                                                                                                                           
    token_acc = calculate_token_accuracy(expected, predicted)                                                                                                                                                                                  
    error = 1 - (token_acc / 100)                                                                                                                                                                                                              
    if is_exact:                                                                                                                                                                                                                               
        exact_matches += 1                                                                                                                                                                                                                     
    total_token_accuracy += token_acc                                                                                                                                                                                                          
    token_accuracies.append(token_acc)                                                                                                                                                                                                         
    errors.append(error)                                                                                                                                                                                                                       
    if (i + 1) % 10 == 0:                                                                                                                                                                                                                      
        print(f"Processed {i+1}/{total}...")                                                                                                                                                                                                   
                                                                                                                                                                                                                                                 
exact_match_pct = (exact_matches / total) * 100                                                                                                                                                                                                
avg_token_acc = total_token_accuracy / total                                                                                                                                                                                                   
mean_error = sum(errors) / len(errors)                                                                                                                                                                                                         
mse = sum(e ** 2 for e in errors) / len(errors)                                                                                                                                                                                                
rmse = math.sqrt(mse)                                                                                                                                                                                                                          
                                                                                                                                                                                                                                                 
mean_acc = sum(token_accuracies) / len(token_accuracies)                                                                                                                                                                                       
ss_tot = sum((acc - mean_acc) ** 2 for acc in token_accuracies)                                                                                                                                                                                
ss_res = sum((acc - 100) ** 2 for acc in token_accuracies)                                                                                                                                                                                     
r_squared = 1 - (ss_res / ss_tot) if ss_tot != 0 else 0                                                                                                                                                                                        
                                                                                                                                                                                                                                                 
print("\n" + "=" * 60)                                                                                                                                                                                                                         
print("EVALUATION SUMMARY - V2")                                                                                                                                                                                                               
print("=" * 60)                                                                                                                                                                                                                                
print(f"Exact Match Accuracy:   {exact_match_pct:.2f}%")                                                                                                                                                                                       
print(f"Mean Token Accuracy:    {avg_token_acc:.2f}%")                                                                                                                                                                                         
print(f"Mean Error:             {mean_error:.4f}")                                                                                                                                                                                             
print(f"MSE:                    {mse:.4f}")                                                                                                                                                                                                    
print(f"RMSE:                   {rmse:.4f}")                                                                                                                                                                                                   
print(f"R-Squared:              {r_squared:.4f}")                                                                                                                                                                                              
print(f"Total Examples:         {total}")                                                                                                                                                                                                      
print(f"Exact Matches:          {exact_matches}") 

EVALUATING V2 ON 50 TEST EXAMPLES
Processed 10/50...
Processed 20/50...
Processed 30/50...
Processed 40/50...
Processed 50/50...

EVALUATION SUMMARY - V2
Exact Match Accuracy:   10.00%
Mean Token Accuracy:    42.38%
Mean Error:             0.5762
MSE:                    0.4586
RMSE:                   0.6772
R-Squared:              -2.6215
Total Examples:         50
Exact Matches:          5


In [68]:
 # Push the model                                                                                                                                                                                                                               
model.push_to_hub("Archanacreates/deepseek-sql-lora")                                                                                                                                                                                           
                                                                                                                                                                                                                                                 
  # Push tokenizer                                                                                                                                                                                                                               
tokenizer.push_to_hub("Archanacreates/deepseek-sql-lora")                                                                                                                                                                                       
                                                                                                                                                                                                                                                 
print("Model uploaded to Hugging Face!")                                                                                                                                                                                                       
                                          

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Model uploaded to Hugging Face!
